Set up up the Spark master
  - "local" for local execution
  - "local[*]" for local execution using all core
  - "spark://spark-master:7077" to connect to the Spark master running on the docker container

For the first two options you will to set up a Python environment and install pyspark.

In [1]:
master="spark://spark-master:7077"

Sequential word count

In [2]:
def seq_word_count(filename: str) -> dict:
    counts = {}
    with open(filename) as f:
        for line in f:
            for word in line.split():
                counts[word] = counts[word]+1 if word in counts else 1
    return {k: v for k, v in sorted(counts.items(), key=lambda x: x[1], reverse=True)}


Running for the Hamlet text

In [3]:
filename = "/data/lab01/hamlet.txt"
topK = 10
result = seq_word_count(filename)
dict(list(result.items())[:topK])

{'the': 988,
 'and': 693,
 'of': 621,
 'to': 604,
 'I': 513,
 'a': 450,
 'my': 441,
 'in': 387,
 'HAMLET': 378,
 'you': 356}

Spark Version.
Setting the Spark session

In [4]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
              .master(master) \
             .appName('word_count') \
             .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/15 15:20:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Base RDD API version

In [5]:
def rdd_word_count(filename: str) -> pyspark.RDD:
    text_file = spark.sparkContext.textFile(filename)
    counts = text_file.flatMap(lambda line: line.split(" ")) \
        .map(lambda word: (word, 1)) \
        .reduceByKey(lambda x, y: x + y)
    return (counts.map(lambda i: (i[1], i[0]))
            .sortByKey(ascending=False)
            .map(lambda i: (i[1], i[0])))

result = rdd_word_count(filename)
result.take(topK)

[('the', 988),
 ('and', 693),
 ('of', 621),
 ('to', 604),
 ('I', 513),
 ('a', 450),
 ('my', 441),
 ('in', 387),
 ('HAMLET', 378),
 ('you', 356)]

PySpark SQL API version


In [6]:
from pyspark.sql.functions import split, explode, col

def sql_word_count(filename: str):
    text_file = spark.read.text(filename)
    words_df = text_file.withColumn("word", explode(split(col("value"), " ")))
    return words_df.groupBy("word").count().orderBy("count", ascending=False)

result = sql_word_count(filename)
result.show(topK)

+------+-----+
|  word|count|
+------+-----+
|   the|  988|
|   and|  693|
|    of|  621|
|    to|  604|
|     I|  513|
|     a|  450|
|    my|  441|
|    in|  387|
|HAMLET|  378|
|   you|  356|
+------+-----+
only showing top 10 rows
